In [ ]:
import pandas as pd


df = pd.read_csv(
    "/kaggle/input/final-main-dataset/final_main_dataset.tsv",  
    sep="\t"
)


df = df.rename(columns={
    "path": "audio_file",
    "sentence": "text"
})

print(df.head())


                                           client_id  \
0  e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...   
1  e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...   
2  e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...   
3  e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...   
4  e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...   

                     audio_file  \
0  common_voice_ur_31771683.mp3   
1  common_voice_ur_31771684.mp3   
2  common_voice_ur_31771685.mp3   
3  common_voice_ur_31771730.mp3   
4  common_voice_ur_31771732.mp3   

                                                text  up_votes  down_votes  \
0                 کبھی کبھار ہی خیالی پلاو بناتا ہوں         2           0   
1                  اور پھر ممکن ہے کہ پاکستان بھی ہو         2           1   
2                      یہ فیصلہ بھی گزشتہ دو سال میں         2           0   
3                     ان کے بلے بازوں کے سامنے ہو گا         3           0   
4  آبی جانور میں بطخ بگلا اور دُوسْرا آبی پرندہ ش...         3

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/final-main-dataset/final_main_dataset.tsv",
    sep="\t"
)

print(df.columns)


Index(['client_id', 'path', 'sentence', 'up_votes', 'down_votes', 'age',
       'gender', 'accents', 'variant', 'locale', 'segment'],
      dtype='object')


In [3]:
import pandas as pd
from datasets import Dataset, Audio
import os
import soundfile as sf

tsv_path = "/kaggle/input/final-main-dataset/final_main_dataset.tsv"  
audio_dir = "/kaggle/input/final-main-dataset/limited_wav_files"      

# Load TSV
df = pd.read_csv(tsv_path, sep="\t")

# Rename columns
df = df.rename(columns={"path": "audio_file", "sentence": "text"})

# Fix filenames to match actual WAVs
df["audio_file"] = df["audio_file"].str.replace(".mp3", ".wav")

# Convert to HF Dataset
dataset = Dataset.from_pandas(df)

# Load audio safely
def load_audio(batch):
    filename = os.path.basename(batch["audio_file"])
    audio_path = os.path.join(audio_dir, filename)
    
    if not os.path.isfile(audio_path):
        print(f"File does not exist: {audio_path}")
        batch["audio"] = None
        return batch
    
    try:
        audio, sr = sf.read(audio_path)
        batch["audio"] = {"array": audio, "sampling_rate": sr}
    except Exception as e:
        print(f"Skipping {audio_path}: {e}")
        batch["audio"] = None
    return batch

dataset = dataset.map(load_audio)
dataset = dataset.filter(lambda x: x["audio"] is not None)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print("Dataset ready. Number of examples:", len(dataset))


/Users/apple/Desktop/tts/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/pr/vq55r_6d7c5_54lzb3x3vl8h0000gn/T/ipykernel_2697/1976670687.py:17: FutureWarning: The default value of regex will change from True to False in a future version.
  df["audio_file"] = df["audio_file"].str.replace(".mp3", ".wav")
Filter: 100%|██████████| 20000/20000 [09:52<00:00, 33.77 examples/s]


ImportError: To support encoding audio data, please install 'torchcodec'.

In [ ]:
# -------------------------------
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

In [ ]:
 
# -------------------------------
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1367.95it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [ ]:
def prepare_dataset(batch):
    # audio → log-mel
    batch["input_features"] = processor(
        batch["audio"]["array"],
        sampling_rate=16000
    ).input_features[0]

    
    labels = processor.tokenizer(
        batch["text"],
        truncation=True,
        max_length=processor.tokenizer.model_max_length
    ).input_ids

    batch["labels"] = labels
    return batch

train_dataset = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names
)

test_dataset = test_dataset.map(
    prepare_dataset,
    remove_columns=test_dataset.column_names
)


Map: 100%|██████████| 2000/2000 [00:33<00:00, 59.82 examples/s]


In [ ]:
import torch

class WhisperDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        labels = labels_batch["input_ids"]
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        batch["labels"] = labels
        return batch


In [ ]:

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,

    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

data_collator = WhisperDataCollator(processor)

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-urdu-fast",
    per_device_train_batch_size=1,     
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_steps=100,                     

    learning_rate=1e-4,
    warmup_steps=0,

   
    save_strategy="no",
    logging_steps=50,
    report_to="none",

    predict_with_generate=False,
    generation_max_length=128,

    remove_unused_columns=False,
)


# ------------------------------------
# Trainer
# ------------------------------------
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

trainer.train()


trainer.save_model("./whisper-urdu")
processor.save_pretrained("./whisper-urdu")


Step,Training Loss
50,0.948432
100,0.656597


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.21s/it]


['./whisper-urdu-fast/processor_config.json']

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Union
import torch

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: WhisperProcessor
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = torch.tensor([f["input_features"] for f in features], dtype=torch.float32)
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return {"input_features": input_features, "labels": labels}

data_collator = DataCollatorSpeechSeq2Seq(processor=processor)

In [ ]:
from transformers import Seq2SeqTrainingArguments

output_dir = "./whisper-urdu-model"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,  # adjust based on your Mac RAM
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=False  # set True if you have GPU support
)



In [ ]:
print("Starting training...")
trainer.train()

Starting training...


/Users/apple/Desktop/tts/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.410286
20,0.505777
30,0.918446
40,0.441580
50,0.527043
60,0.459591
70,0.692287
80,0.759845
90,0.577970
100,0.575634


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.80s/it]


ValueError: Labels' sequence length 1024 cannot exceed the maximum allowed length of 448 tokens.

In [64]:
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print("Model saved to", output_dir)


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.66s/it]


Model saved to ./whisper-urdu-model
